# Publish raster tile layer items using vector tile content

This sample exemplifies a workflow for publishing a Map Service serving raster tiles that are sourced from vector tile content. This enables organizations using technology that cannot render vector tiles to make use of updated content that is stored as vector tiles. 

Please see the [Share raster tile layers from vector tile content](https://doc.esri.com/en/arcgis-enterprise/latest/administer/generate-map-image-layers-from-vector-content.html?pivots=os-windows) help topic for full details.

## Step 0. Make a connection to your ArcGIS Enterprise portal 

In [ ]:
from pathlib import Path
from arcgis.layers import MapImageLayer
from arcgis.geometry import Polygon
import json
from arcgis.gis import GIS, ItemProperties, ItemTypeEnum


In [ ]:
# Connect to your GIS
# Create a connection to your portal for publishing
gis=GIS(url="https://<yourEnteprisePortal>", username="<username>", password="<password>")

## Step 1. Select the hosting/federated server to which you would like to publish the map image layer.  

In [ ]:
server_list = gis.admin.federation.servers
server_list

In [ ]:
server_id = None
for server in server_list['servers']:
    if server['serverRole'] == 'FEDERATED_SERVER': # HOSTING_SERVER
        server_id = server['id']
        break
if server_id is None:
    raise Exception("Cannot find FEDERATED_SERVER on this Enterprise configuration")
server_id

> **NOTE:** If you 

## Step 2. Create and Register the data stores containing vector content

1. Reference a location on file system where vector tile layer content resides. Make sure the server install owner has READ access to this location.  
2. Create a folder and data item dictionary and add content to the ArcGIS Enterprise portal.
3. Register the data store item with the ArcGIS Enteprise server component whose server_id value was obtained in step 1. 
> **note:** Go to step 4a if a data store with vector content has already been registered. 

In [ ]:
# Location on file system where vector tile layer content resides
folder_path = Path(r"\\<unc_Or_Local_path\to\location\where\VectorContent\resides>")
folder_path.exists()

In [ ]:
# Create a user managed Folder data store of the location where source vector tile content & style root resides
ds_folder = gis.content.folders.create(folder="<my_datastore_items_folder_name>")
ds_config = {
    "type": "folder",
    "path": "/fileShares/<myVectorsourceData>",
    "info":{
        "isManaged": False,
        "dataStoreConnectionType": "shared",
        "path": str(folder_path)
    }
}
txt = json.dumps(ds_config)

In [ ]:
# Add data store as an item to my portal
ds_item = ds_folder.add(item_properties=ItemProperties(
    title="<my_Source_vector_folder_data_store_name>",
    item_type=ItemTypeEnum.DATA_STORE,# must be this
    tags="Source Vector Tile content",
    text=txt
    )
).result()      
ds_item

In [ ]:
# Registers the Data Store to the to the server in ArcGIS Enterprise selected in step 1
gis.datastore.register(item=ds_item, server_id=server_id)

## Step 3. Create and Register CacheStore

1. Assing a variable to a location on your filesystem if craeating a folder data store or, ensure you have the information to reference an objectstore
in the cloud. Make sure the server install owner has READ & WRITE access to this location.
2. Create a Python dictionary describing the data item, whether it's a folder datastore or objectstore datastore. 
3. Register the data store to the to the ArcGIS Enterprise server whose server_id was obtained in step 1.
This location would be used as CACHE Store to generate raster tiles for the map service.
> **Note:** Go to step 4b, if a data store for the cache has already been registered. 

In [ ]:
# path to location on file system where raster cache would be created
cpath = Path(r"\\UNC_orLocal\path_to_raster_cache_location")

> **Note:** Use either step 3a or 3b depending upon which type of data store you are creating

### Data item configuration for a folder data store

In [ ]:
# Step 3a.  Create a user managed cache data store item configuration for location on file system where raster cache would be created
cds_config = {
    "type": "folder",
    "path": "/fileShares/<myCacheDatastore_name>",
    "info":{
        "isManaged": False,
        "dataStoreConnectionType": "shared",
        "purposes": ["map-tile"],
        "path": str(cpath)
    }
}

### Data item configuration for an objectstore data store

In [ ]:
# Step 3b. Create a user managed cache data store item configuration using an object store by AWS Cloudprovider where raster cache would be created.
                    
cds_config = {
    "path": "/cloudStores/<myCacheDatastore_name>",
    "type": "objectStore",
    "provider": "amazon",
    "info": {
        "isManaged": False,
        "purposes": ["map-tile"],
        "connectionString": "{\"accessKeyId\":\"<access_key_value>\",\"secretAccessKey\":\"<secret_access_value>\",\"regionEndpointUrl\":\"https://s3.us-west-2.amazonaws.com\",\"defaultEndpointsProtocol\":\"https\",\"credentialType\":\"accesskey\",\"region\":\"us-west-2\"}",
        "objectStore": "bucket/folder"
    }
}
ctxt = json.dumps(cds_config)

### Add the data store item to the ArcGIS Enteprise portal

In [ ]:
# Add data store as an item to my portal
cds_item = ds_folder.add(item_properties=ItemProperties(
    title="<my_cachedatastore_folderName>",
    item_type=ItemTypeEnum.DATA_STORE, # must be this
    tags="Cache Store output",
    text=cds_config
    )
).result()
cds_item

### Register the data store item with ArcGIS Enteprise server

In [ ]:
# Registers the Data Store to the to the server in ArcGIS Enterprise selected in step 1
gis.datastore.register(item=cds_item, server_id=server_id)

## Step 4. **Only** if using previously registered Data stores item information

If you have already registered the data stores with the server, get the respective data store item id values to use in the service configuration.

### Step 4a. Item id value for data store item referencing vector content

In [ ]:
ds_item= "<ItemID>"

### Step 4b. Item id value for the data store item referencing the cache 

In [ ]:
cds_item= "<ItemID>"

## Step 5. Configure the parameters for the tile layer and publish it.

1. Provide path to the vector style resource in folder Data Store. This style would be used to create the raster tile layer item 
2. Define the tiling scheme dictionary for the raster tile layer.
3. Provide values for your service name and cache directory folder in the cache data store 
4. Configure properties for your map service.
5. Publish the "Map Image Layer" using vector style layer content.

In [ ]:
# 5.1. Provide path to the vector style resource located in the Data store registered at step 2 or step 4a. 
# Please ensure style root resource has correct references to the Tile root resource being used.
VectorStyleRoot= Path(r"Esri_Vector_Basemap_DA2026_Install.vtiles\coloredpencil\resources\styles\root.json")

In [ ]:
# Validates the vector style exists in the folder datastore registered at step 2 or 4a
fullPath_VectorStyleRoot = folder_path / VectorStyleRoot
fullPath_VectorStyleRoot.exists()

In [ ]:
# 5.2 Create tiling scheme dictionary for the mapservice cache
scheme={
    "rasterCacheSchema": {
        "tileInfo": {
            "spatialReference": {
                "wkid": 102100,
                "latestWkid": 3857
            },
            "origin": {
                "x": -20037508.342787,
                "y": 20037508.342787
            },
            "rows": 256,
            "cols": 256,
            "dpi": 96,
            "lods": [
            {
                "level": 0,
                "resolution": 156543.033928,
                "scale": 591657527.591555
            },
            {
                "level": 1,
                "resolution": 78271.5169639999,
                "scale": 295828763.795777
            },
            {
                "level": 2,
                "resolution": 39135.7584820001,
                "scale": 147914381.897889
            },
            {
                "level": 3,
                "resolution": 19567.8792409999,
                "scale": 73957190.948944
            },
            {
                "level": 4,
                "resolution": 9783.93962049996,
                "scale": 36978595.474472
            },
            {
                "level": 5,
                "resolution": 4891.96981024998,
                "scale": 18489297.737236
            },
            {
                "level": 6,
                "resolution": 2445.98490512499,
                "scale": 9244648.868618
            },
            {
                "level": 7,
                "resolution": 1222.99245256249,
                "scale": 4622324.434309
            },
            {
                "level": 8,
                "resolution": 611.49622628138,
                "scale": 2311162.217155
            },
            {
                "level": 9,
                "resolution": 305.748113140558,
                "scale": 1155581.108577
            },
            {
                "level": 10,
                "resolution": 152.874056570411,
                "scale": 577790.554289
            },
            {
                "level": 11,
                "resolution": 76.4370282850732,
                "scale": 288895.277144
            },
            {
                "level": 12,
                "resolution": 38.2185141425366,
                "scale": 144447.638572
            },
            {
                "level": 13,
                "resolution": 19.1092570712683,
                "scale": 72223.819286
            },
            {
                "level": 14,
                "resolution": 9.55462853563415,
                "scale": 36111.909643
            },
            {
                "level": 15,
                "resolution": 4.77731426794937,
                "scale": 18055.954822
            },
            {
                "level": 16,
                "resolution": 2.38865713397468,
                "scale": 9027.977411
            },
            {
                "level": 17,
                "resolution": 1.19432856685505,
                "scale": 4513.988705
            },
            {
                "level": 18,
                "resolution": 0.597164283559817,
                "scale": 2256.994353
            },
            {
                "level": 19,
                "resolution": 0.298582141647617,
                "scale": 1128.497176
            },
            {
                "level": 20,
                "resolution": 0.14929107082380833,
                "scale": 564.248588
            },
            {
                "level": 21,
                "resolution": 0.074645535411904163,
                "scale": 282.124294
            },
            {
                "level": 22,
                "resolution": 0.037322767705952081,
                "scale": 141.062147
            },
            {
                "level": 23,
                "resolution": 0.018661383852976041,
                "scale": 70.5310735
            }
        ]
        },
        "storageInfo": {
            "packetSize": 128,
            "storageFormat": "esriMapCacheStorageModeCompactV2"
        },
        "tileImageInfo": {
            "format": "JPEG",
            "compressionQuality": 75
        },
		"wmtsScaleFactor": 0.00028000000000003521
    }
}

In [ ]:
# 5.3. Provide values for your service name & cache directory folder in CacheStore 
desiredServiceName = "<desiredName>"
cacheFolderName = "/<desiredFolderName>"

In [ ]:
# 5.4. Create a dictionary for your service configuration
service_conf={
    "serviceName": desiredServiceName,
    "type": "MapServer",
    "properties": {
        "sourceVtDatastoreId": ds_item.id,
        "sourceVtStylePath": str(VectorStyleRoot),
        "pathInCachedStore": cacheFolderName,
        "cacheStoreId": cds_item.id,
        "exportTilesAllowed":"true" ,# set it to false if you would like disable offline capabilities
        "minScale": 691657527,
        "maxScale": 564
    }
}
service_conf

In [ ]:
ds = gis.datastore

In [ ]:
## 5.5. Publish the vector style as a tile layer in the ArcGIS Enteprise portal. The tile layer content is 
## located in the data store managed by you.
## The server_id argument must be the same value used to register both the vector tile and cache datastore items.

mil_item = ds.publish(
    config=service_conf, 
    server_id=server_id,
    folder="",
    description="CachedmapLayer from vector style by python API", 
    tags="Basemap layer using Vector tiles",
    options=scheme,
    overwrite="false"
).result()

## Tile layer is published and now is serving content directly from the vector data store 

In [ ]:
mil_item[0]

## Step 6. Pre-generate raster tiles for frequently visited scale levels and geographic areas of the Tile Layer 

> **Note:** ArcGIS Enterprise portal Tile Layer item content is managed in the ArcGIS API for Python using the [MapImageLayer](/python/latest/api-reference/arcgis.layers.toc.html#mapimagelayer) class.

In [ ]:
mimg_lyr = MapImageLayer.fromitem(
    item=mil_item[0]
)
ent_mapimg_mgr = mimg_lyr.manager

In [ ]:
# Define the Level of details at which you would like the cache to be built
lod=[18489297.737236001, 9244648.8686180003, 4622324.4343090001, 2311162.2171550002 ]

In [ ]:
# Update the values below to define the Extents at which you would like the cache to be built
extent_poly = Polygon(
    iterable={
        "xmin": -20037507.0671618,
        "ymin": -20037507.0671618,
        "xmax": 20037507.0671618,
        "ymax": 20037507.0671618,
        "spatialReference": {"wkid":102100, "latestWkid": 3857}
    }
)

In [ ]:
# Pre-generate tiles for the desired extent of the tile layer
cache_job_poly = ent_mapimg_mgr.build_cache(
    levels=lod,
    extent=extent_poly
) 
cache_job_poly

## Step 7. Overwrite the map image layer when the source vector style or tiles gets updated
* Use this step when the vector tile source content or style resource is updated. 

In [ ]:
# Provide path to the new style in the folder datastore where vector tile contnet is located.
new_style_path = Path(r"Esri_Vector_Basemap_DA2026_Install.vtiles\DAstyles\canvasdarkbase\resources\root.json")

In [ ]:
# Create dictonary for the mapimage layer. Note the service name, vectordatastoreID, CacheSTore ID, tiling scheme, pathInCacheStore cannot be changed.
service_conf={
    "serviceName": desiredServiceName,
    "type": "MapServer",
    "properties": {
        "sourceVtDatastoreId": ds_item.id,
        "sourceVtStylePath": str(new_style_path),
        "pathInCachedStore": cacheFolderName,
        "cacheStoreId": cds_item.id,
        "exportTilesAllowed":"true",
        "minScale": 591657527,
        "maxScale": 564
    }
}
service_conf

In [ ]:
# Overwrite and oublish the new style path as mapservice
mil_item = ds.publish(
    config=service_conf, 
    server_id=server_id,
    folder="/",
    description="MapImageLayer by python API after overwrite", 
    tags="pythonAPI test, overwriting style",
    overwrite=True
).result()


In [ ]:
mil_item[0]